## Apache Log Analysis using PYSPARK
### Dataset:NASA Web server logs

#### Creating Spark session:

In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Log_Analysis").config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem").getOrCreate()
print("Spark created successfully")

Spark created successfully


In [2]:
spark

In [3]:
logs=spark.read.text("nasa.log")
logs.show(3,truncate=False)

+--------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                     |
+--------------------------------------------------------------------------------------------------------------------------+
|in24.inetnebr.com - - [01/Aug/1995:00:00:01 -0400] "GET /shuttle/missions/sts-68/news/sts-68-mcc-05.txt HTTP/1.0" 200 1839|
|uplherc.upl.com - - [01/Aug/1995:00:00:07 -0400] "GET / HTTP/1.0" 304 0                                                   |
|uplherc.upl.com - - [01/Aug/1995:00:00:08 -0400] "GET /images/ksclogo-medium.gif HTTP/1.0" 304 0                          |
+--------------------------------------------------------------------------------------------------------------------------+
only showing top 3 rows


In [4]:
# total no. of entries in log file
logs.count()

1569898

### Creating Dataframe:

#### Regex is used to extract structured fields from raw log entries

In [5]:
from pyspark.sql.functions import regexp_extract

In [6]:
pattern = r'^(\S+)\s+\S+\s+\S+\s+\[([^\]]+)\]\s+"(GET|POST|HEAD|PUT|DELETE|OPTIONS|PATCH)\s+([^"]+)"\s+(\d{3})\s+(\S+)$'

In [7]:
test_df=logs.select(
    regexp_extract("value",pattern,1).alias("host")
)
test_df.show(5)

+-----------------+
|             host|
+-----------------+
|in24.inetnebr.com|
|  uplherc.upl.com|
|  uplherc.upl.com|
|  uplherc.upl.com|
|  uplherc.upl.com|
+-----------------+
only showing top 5 rows


In [8]:
failed_lines=test_df.filter(test_df.host=="").count()
failed_lines

28

In [9]:
matched_lines=logs.count()-failed_lines
matched_lines

1569870

In [10]:
failed_logs=logs.filter(
    regexp_extract("value",pattern,1)==""
)
failed_logs.show(20,truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                               |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|198.213.130.253 - - [03/Aug/1995:11:29:02 -0400] "GET /shuttle/missions/sts-34/mission-sts-34.html"><IMG images/ssbuv1.gif SRC="images/small34p.gif/ HTTP/1.0" 404 -|
|userp2.snowhill.com - - [05/Aug/1995:14:57:06 -0400] "GET / " HTTP/1.0" 200 7034                                                                                    |
|jhanley.doe.state.la.us - - [08/Aug/1995:11:24:31 -0400] "GET /ksc.html" HTTP/1.0" 404 -                                                                            

#### Log entries are parsed into structured columns

In [11]:
parsed_logs=logs.select(
    regexp_extract("value",pattern,1).alias("host"),
    regexp_extract("value",pattern,2).alias("timestamp"),
    regexp_extract("value",pattern,3).alias("method"),
    regexp_extract("value",pattern,4).alias("url"),
    regexp_extract("value",pattern,5).alias("status"),
    regexp_extract("value",pattern,6).alias("size"),
)
parsed_logs.show(10,truncate=False)

+---------------------------+--------------------------+------+--------------------------------------------------------+------+-----+
|host                       |timestamp                 |method|url                                                     |status|size |
+---------------------------+--------------------------+------+--------------------------------------------------------+------+-----+
|in24.inetnebr.com          |01/Aug/1995:00:00:01 -0400|GET   |/shuttle/missions/sts-68/news/sts-68-mcc-05.txt HTTP/1.0|200   |1839 |
|uplherc.upl.com            |01/Aug/1995:00:00:07 -0400|GET   |/ HTTP/1.0                                              |304   |0    |
|uplherc.upl.com            |01/Aug/1995:00:00:08 -0400|GET   |/images/ksclogo-medium.gif HTTP/1.0                     |304   |0    |
|uplherc.upl.com            |01/Aug/1995:00:00:08 -0400|GET   |/images/MOSAIC-logosmall.gif HTTP/1.0                   |304   |0    |
|uplherc.upl.com            |01/Aug/1995:00:00:08 -0400|GET   

#### malformed logentries are removed

In [12]:
clean_logs=parsed_logs.filter(parsed_logs.host!="")

#### status & size fields are converted to integer datatypes for analysis

In [13]:
from pyspark.sql.functions import col,expr,coalesce,lit
clean_logs=clean_logs.withColumn("status",expr("try_cast(status as int)"))
clean_logs=clean_logs.withColumn("size",expr("try_cast(size as int)"))
clean_logs=clean_logs.withColumn(
    "size",coalesce(col("size"),lit(0))
)

In [14]:
clean_logs.printSchema()

root
 |-- host: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- status: integer (nullable = true)
 |-- size: integer (nullable = false)



#### Malformed entries were removed duriong preprocessing

In [15]:
clean_logs.count()

1569870

In [16]:
clean_logs.show(10,truncate=False)

+---------------------------+--------------------------+------+--------------------------------------------------------+------+-----+
|host                       |timestamp                 |method|url                                                     |status|size |
+---------------------------+--------------------------+------+--------------------------------------------------------+------+-----+
|in24.inetnebr.com          |01/Aug/1995:00:00:01 -0400|GET   |/shuttle/missions/sts-68/news/sts-68-mcc-05.txt HTTP/1.0|200   |1839 |
|uplherc.upl.com            |01/Aug/1995:00:00:07 -0400|GET   |/ HTTP/1.0                                              |304   |0    |
|uplherc.upl.com            |01/Aug/1995:00:00:08 -0400|GET   |/images/ksclogo-medium.gif HTTP/1.0                     |304   |0    |
|uplherc.upl.com            |01/Aug/1995:00:00:08 -0400|GET   |/images/MOSAIC-logosmall.gif HTTP/1.0                   |304   |0    |
|uplherc.upl.com            |01/Aug/1995:00:00:08 -0400|GET   

#### HTTP method distribution

In [17]:
clean_logs.groupBy("method").count().orderBy("count",ascending=False).show()

+------+-------+
|method|  count|
+------+-------+
|   GET|1565794|
|  HEAD|   3965|
|  POST|    111|
+------+-------+



#### Calculating the avg size of responses served by the NASA web server

In [18]:
clean_logs.show(5)
clean_logs.count()

+-----------------+--------------------+------+--------------------+------+----+
|             host|           timestamp|method|                 url|status|size|
+-----------------+--------------------+------+--------------------+------+----+
|in24.inetnebr.com|01/Aug/1995:00:00...|   GET|/shuttle/missions...|   200|1839|
|  uplherc.upl.com|01/Aug/1995:00:00...|   GET|          / HTTP/1.0|   304|   0|
|  uplherc.upl.com|01/Aug/1995:00:00...|   GET|/images/ksclogo-m...|   304|   0|
|  uplherc.upl.com|01/Aug/1995:00:00...|   GET|/images/MOSAIC-lo...|   304|   0|
|  uplherc.upl.com|01/Aug/1995:00:00...|   GET|/images/USA-logos...|   304|   0|
+-----------------+--------------------+------+--------------------+------+----+
only showing top 5 rows


1569870

In [26]:
with open("clean_nasa_logs.csv", "w", encoding="utf-8") as f:
    
    # write header
    f.write(",".join(clean_logs.columns) + "\n")
    
    # write rows
    for row in clean_logs.collect():
        f.write(",".join([str(x) for x in row]) + "\n")

In [27]:
# header = ",".join(clean_logs.columns)

# clean_logs.coalesce(1).rdd.map(lambda row: ",".join([str(x) for x in row])).saveAsTextFile("clean_nasa_logs_data")
# spark.sparkContext.parallelize([header]).saveAsTextFile("clean_nasa_logs_header")

In [28]:
# clean_logs.coalesce(1).write.mode("overwrite").option("header", True).csv("clean_nasa_logs")

In [ ]:
# output_path = r"C:\Users\Lanka\Documents\Log_Project\clean_nasa_logs_output"
# output_path

In [ ]:
# clean_logs.coalesce(1).write.mode("overwrite").csv(
#     r"C:\Users\Lanka\Documents\Log_Project\clean_nasa_logs",
#     header=True
# )
# clean_logs.coalesce(1).write.mode("overwrite").parquet("clean_nasa_logs_parquet")

In [ ]:
# spark.conf.set("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")

In [ ]:
# clean_logs.printSchema()
# clean_logs.count()

In [ ]:
# import os
# os.getcwd()